# Surface de volatilité implicite — Guide

Ce notebook est l'outil d'onboarding du moteur de pricing/risk options. À l'issue de cette lecture,
vous serez en mesure de **lire et interpréter tous les outputs du moteur** : prix, Greeks, smile par
maturité, surface 3D, violations calendaires.

**Prérequis :** culture marchés financiers.

**Structure :**
- **Bloc 1** — Options et volatilité implicite : les fondations
- **Bloc 2** — Les Greeks : tableau de bord d'une position options
- **Bloc 3** — La tranche : lire un smile à maturité fixée
- **Bloc 4** — La surface complète : empiler les tranches en 3D

---
Run : `uv run --group notebooks jupyter lab notebooks/vol_surface_pedagogique.ipynb`

## Vocabulaire de base

- **ATM / ITM / OTM** — position du strike vs sous-jacent. Call **ATM** : strike ≈ spot ; **ITM** : valeur intrinsèque (spot > strike) ; **OTM** : pas encore (spot < strike). Inverse pour un put.
- **Forward `F`** — prix à terme du sous-jacent pour l'échéance (spot + portage taux/dividendes). La vraie référence « à la monnaie », pas le spot.
- **Log-moneyness `k = ln(K/F)`** — `0` à la monnaie, `< 0` côté puts, `> 0` côté calls. Rend les smiles comparables d'une échéance à l'autre.

In [ ]:
# ── Setup : imports, Black-Scholes, fonctions smile/surface, template Plotly ──
import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# ── Palette & template ────────────────────────────────────────────────────────
C = {
    "blue":    "#2563EB", "teal":   "#0D9488", "violet": "#7C3AED",
    "amber":   "#D97706", "red":    "#DC2626",  "green":  "#16A34A",
    "indigo":  "#4F46E5", "slate9": "#0F172A",  "slate6": "#475569",
    "slate4":  "#94A3B8", "slate1": "#F1F5F9",  "white":  "#FFFFFF",
}
DISCRETE = [C["blue"], C["teal"], C["violet"], C["amber"], C["red"], C["green"]]
FONT = "Inter, IBM Plex Sans, -apple-system, sans-serif"
SURFACE_CS = [
    [0.00, "#1E3A5F"], [0.25, "#2563EB"], [0.50, "#0D9488"],
    [0.75, "#D97706"], [1.00, "#DC2626"],
]
CAMERA = dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))
ASPECT = dict(x=1.5, y=1.2, z=0.8)

_ax = dict(
    showgrid=True, gridcolor=C["slate4"], gridwidth=0.5,
    zeroline=False, linecolor=C["slate4"], linewidth=1,
    ticks="outside", tickcolor=C["slate4"],
    tickfont=dict(size=11), title_font=dict(size=12, color=C["slate6"]),
)
_tmpl = go.layout.Template()
_tmpl.layout = go.Layout(
    font=dict(family=FONT, size=12, color=C["slate6"]),
    title_font=dict(family=FONT, size=15, color=C["slate9"]),
    paper_bgcolor=C["white"], plot_bgcolor=C["white"],
    colorway=DISCRETE, xaxis=_ax, yaxis=_ax,
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor=C["slate4"],
                borderwidth=1, font=dict(size=11)),
    margin=dict(l=64, r=24, t=80, b=56),
    hoverlabel=dict(bgcolor=C["slate9"], font=dict(color=C["white"], size=12,
                    family=FONT), bordercolor=C["slate9"]),
    hovermode="closest",
)
pio.templates["ped"] = _tmpl
pio.templates.default = "plotly_white+ped"

# ── Grilles communes ─────────────────────────────────────────────────────────
k_grid      = np.linspace(-0.5, 0.5, 80)              # log-moneyness
T_arr       = np.array([30, 60, 90, 180]) / 365        # maturités en années
T_labels    = ["30j", "60j", "90j", "180j"]
T_fine      = np.linspace(1/365, 1.0, 200)

# ── Black-Scholes : prix + 5 Greeks ──────────────────────────────────────────
def bs(S, K, T, r, sigma, flag="call"):
    """Prix BS et cinq Greeks. vega/1% vol, theta/jour calendaire, rho/bp."""
    if T < 1e-8:
        intrinsic = max(S - K, 0.) if flag == "call" else max(K - S, 0.)
        return dict(price=intrinsic, delta=0., gamma=0., vega=0., theta=0., rho=0.)
    sqT = np.sqrt(T)
    d1  = (np.log(S / K) + (r + .5 * sigma**2) * T) / (sigma * sqT)
    d2  = d1 - sigma * sqT
    disc = np.exp(-r * T)
    nd1  = norm.pdf(d1)
    if flag == "call":
        price = S * norm.cdf(d1) - K * disc * norm.cdf(d2)
        delta = norm.cdf(d1)
        raw_theta = -S * nd1 * sigma / (2 * sqT) - r * K * disc * norm.cdf(d2)
        raw_rho   =  K * T * disc * norm.cdf(d2)
    else:
        price = K * disc * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1.
        raw_theta = -S * nd1 * sigma / (2 * sqT) + r * K * disc * norm.cdf(-d2)
        raw_rho   = -K * T * disc * norm.cdf(-d2)
    gamma = nd1 / (S * sigma * sqT)
    vega  = S * sqT * nd1 * 0.01       # par +1% de vol
    theta = raw_theta / 365.           # par jour calendaire
    rho   = raw_rho * 0.0001           # par +1bp
    return dict(price=price, delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)

# ── Smile paramétrique IV(k) = ATM + skew*k + conv*k² ───────────────────────
def smile_iv(k, atm, skew, conv):
    return np.clip(atm + skew * k + conv * k**2, 0.01, 5.0)

# ── Term structures ───────────────────────────────────────────────────────────
def ts_backwardation(T): return 0.22 + 0.43 * np.exp(-8. * T)
def ts_flat(T):          return np.full_like(np.atleast_1d(T), .25, dtype=float)
def ts_contango(T):      return 0.35 - 0.23 * np.exp(-4. * T)

# ── Surface (n_T × n_k) ───────────────────────────────────────────────────────
def build_surface(T_arr, k_arr, atm_fn, base_skew, base_conv,
                  skew_fade=0., conv_fade=0.):
    iv = np.zeros((len(T_arr), len(k_arr)))
    for i, T in enumerate(T_arr):
        sk = base_skew * np.exp(-skew_fade * T)
        cv = base_conv * np.exp(-conv_fade * T)
        iv[i] = smile_iv(k_arr, atm_fn(T), sk, cv)
    return iv

# Surfaces pré-calculées
surf_equity = build_surface(T_arr, k_grid, ts_contango,
                             base_skew=-0.60, base_conv=0.40,
                             skew_fade=3.0, conv_fade=2.0)
surf_crypto = build_surface(T_arr, k_grid, ts_backwardation,
                             base_skew=0.00, base_conv=2.50, conv_fade=3.5)
surf_arb    = surf_equity.copy()
surf_arb[1] = surf_equity[0] * 0.55    # violation calendaire slice 60j (sigma^2*T decroit)

KK, TT = np.meshgrid(k_grid, T_arr * 365)
print("✓ Setup OK")

---
# Bloc 1 — Options et volatilité implicite : les fondations

## 1.1 — Le profil de payoff d'une option

Une option est un **droit asymétrique** : l'acheteur ne peut perdre que sa prime, mais son gain
est potentiellement illimité (call) ou borné par le strike (put). Le vendeur encaisse la prime
mais supporte le risque inverse.

In [ ]:
S_range = np.linspace(60, 140, 300)
K, premium_c, premium_p = 100., 5., 4.5

payoffs = {
    ("Long Call",  C["blue"]):   np.maximum(S_range - K, 0) - premium_c,
    ("Short Call", C["red"]):   -(np.maximum(S_range - K, 0) - premium_c),
    ("Long Put",   C["teal"]):   np.maximum(K - S_range, 0) - premium_p,
    ("Short Put",  C["amber"]): -(np.maximum(K - S_range, 0) - premium_p),
}

fig = make_subplots(rows=2, cols=2, subplot_titles=[l for (l, _) in payoffs],
                    vertical_spacing=0.12, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for ((label, color), pnl), (r, c) in zip(payoffs.items(), positions):
    fig.add_trace(go.Scatter(
        x=S_range, y=pnl, mode="lines", line=dict(color=color, width=2.5),
        name=label, showlegend=False,
        hovertemplate=f"<b>{label}</b><br>Spot: %{{x:.1f}}<br>P&L: %{{y:.2f}}<extra></extra>",
    ), row=r, col=c)
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate4"], row=r, col=c)
    fig.add_vline(x=K,  line_dash="dot",  line_color=C["slate4"], row=r, col=c)

fig.update_layout(title="Profils de P&L à expiry — Strike K=100, r=0", height=480,
                  margin=dict(t=90))
fig.update_yaxes(title_text="P&L", row=1, col=1)
fig.update_yaxes(title_text="P&L", row=2, col=1)
fig.update_xaxes(title_text="Spot à expiry", row=2, col=1)
fig.update_xaxes(title_text="Spot à expiry", row=2, col=2)
fig.show()

**Interprétation.** L'asymétrie est la propriété fondamentale : le long call gagne de manière
illimitée si le marché monte, mais sa perte est strictement bornée à la prime payée. C'est
précisément cette asymétrie qui a un prix — et ce prix est entièrement déterminé par la
volatilité attendue du sous-jacent.

## 1.2 — La volatilité comme paramètre de dispersion

Un actif à 60% de vol annuelle peut perdre ou gagner 4% en une seule journée sans que ce soit
statistiquement exceptionnel. Un actif à 10% de vol, non. C'est ce seul paramètre qui détermine
la valeur d'une option, toutes choses égales par ailleurs.

In [ ]:
vol_scenarios = [("σ = 10%  (obligataire)", 0.10, C["teal"]),
                 ("σ = 25%  (actions)",     0.25, C["blue"]),
                 ("σ = 60%  (crypto)",       0.60, C["red"])]
dt = 1/252; N = 252
days = np.arange(N + 1)

paths = {}
# Change DISPERSION_SEED to reroll the (now reproducible) trajectories.
DISPERSION_SEED = 7
for i, (label, sig, _) in enumerate(vol_scenarios):
    rng  = np.random.default_rng(DISPERSION_SEED + i)
    Z    = rng.standard_normal(N)
    path = np.empty(N + 1); path[0] = 100.
    path[1:] = 100. * np.exp(np.cumsum((-0.5*sig**2*dt) + sig*np.sqrt(dt)*Z))
    paths[label] = path

fig = make_subplots(rows=1, cols=2, column_widths=[0.60, 0.40],
                    subplot_titles=("Trajectoires de prix (252 jours)",
                                    "Distribution des rendements journaliers"))
for (label, sig, color), path in zip(vol_scenarios, paths.values()):
    fig.add_trace(go.Scatter(
        x=days, y=path, mode="lines", line=dict(color=color, width=1.5),
        name=label,
        hovertemplate="Jour %{x}<br>Prix: %{y:.1f}<extra></extra>",
    ), row=1, col=1)
    log_rets = np.diff(np.log(path)) * 100
    fig.add_trace(go.Histogram(
        x=log_rets, nbinsx=40, marker_color=color, opacity=0.6,
        name=label, showlegend=False,
        hovertemplate="Rendement: %{x:.1f}%<br>Fréquence: %{y}<extra></extra>",
    ), row=1, col=2)

fig.add_hline(y=100, line_dash="dash", line_color=C["slate4"], row=1, col=1)
fig.update_xaxes(title_text="Jours", row=1, col=1)
fig.update_xaxes(title_text="Rendement journalier (%)", row=1, col=2)
fig.update_yaxes(title_text="Prix", row=1, col=1)
fig.update_layout(title="Trois régimes de volatilité — S₀ = 100, r = 0",
                  height=400, barmode="overlay", margin=dict(t=90))
fig.show()

**Interprétation.** La distribution des rendements journaliers dit tout : à 10% de vol, les
mouvements sont concentrés dans un couloir étroit. À 60%, la distribution s'étale — les queues
sont épaisses, les mouvements extrêmes sont fréquents. Un acheteur d'options sur l'actif à 60%
a beaucoup plus de chances de finir dans la monnaie : la prime doit donc être bien plus élevée.

## 1.3 — Prix d'option en fonction de la volatilité

Pour un call ATM à 3 mois, la relation prix/vol est monotone et quasi-linéaire autour de l'ATM.
**Acheter une option, c'est acheter de la volatilité.** Le strike et la maturité fixent le levier,
mais c'est la vol qui détermine le prix.

In [ ]:
vols = np.linspace(0.01, 1.0, 200)
S, K, T3m, r = 100., 100., 90/365, 0.03

prices_atm  = [bs(S, K,     T3m, r, v)["price"] for v in vols]
prices_otm  = [bs(S, K*1.1, T3m, r, v)["price"] for v in vols]  # OTM call
prices_itm  = [bs(S, K*0.9, T3m, r, v)["price"] for v in vols]  # ITM call

fig = go.Figure()
for label, prices, color in [
    ("ATM  K=100",     prices_atm,  C["blue"]),
    ("OTM  K=110",     prices_otm,  C["amber"]),
    ("ITM   K=90",     prices_itm,  C["teal"]),
]:
    fig.add_trace(go.Scatter(
        x=vols*100, y=prices, mode="lines", line=dict(width=2.5, color=color),
        name=label,
        hovertemplate=f"<b>{label}</b><br>Vol: %{{x:.0f}}%<br>Prix: %{{y:.3f}}<extra></extra>",
    ))

for vol_mark, label_m in [(10, "10%"), (25, "25%"), (60, "60%")]:
    fig.add_vline(x=vol_mark, line_dash="dot", line_color=C["slate4"])
    fig.add_annotation(x=vol_mark, y=0.5, text=label_m, showarrow=False,
                       font=dict(size=10, color=C["slate6"]))

fig.update_layout(
    title="Prix d'un call européen vs volatilité  (S=100, T=90j, r=3%)",
    xaxis_title="Volatilité implicite (%)", yaxis_title="Prix (USD)", height=380,
)
fig.show()

**Interprétation.** L'ATM est la position la plus sensible à la vol — la courbe est presque une
droite : doubler la vol double à peu près le prix. L'OTM est non-linéaire : à faible vol le prix
est quasi-nul (probabilité d'exercice infime), à forte vol il rattrape l'ATM. L'ITM démarre avec
une valeur intrinsèque (K=90 < S=100 → valeur plancher) et devient moins sensible à la vol
marginale. Un trader d'options ne parle jamais du prix en dollars — il parle de la vol. (Réserve : ce raccourci suppose une option couverte en delta ; achetée nue, une option reste avant tout un pari directionnel.)

## 1.4 — L'inversion : du prix de marché à la vol implicite

Si le marché cote un call ATM à 3.80, et que je connais tous les autres paramètres, quelle vol
cela implique-t-il ? C'est l'inversion de Black-Scholes. La vol implicite est **le seul paramètre
d'information contenu dans le prix d'une option** au-delà du observable.

In [ ]:
market_prices_example = {"Prix = 2.50": 2.50, "Prix = 5.00": 5.00, "Prix = 8.50": 8.50}
colors_ex = [C["teal"], C["blue"], C["violet"]]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=vols*100, y=prices_atm, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Prix ATM vs vol",
    hovertemplate="Vol: %{x:.0f}%<br>Prix: %{y:.3f}<extra></extra>",
))

for (label, mkt_price), col in zip(market_prices_example.items(), colors_ex):
    # Vol implicite approx : trouver l'intersection
    idx = np.argmin(np.abs(np.array(prices_atm) - mkt_price))
    impl_vol = vols[idx] * 100
    fig.add_hline(y=mkt_price, line_dash="dash", line_color=col,
                  annotation_text=f"{label}  →  IV ≈ {impl_vol:.0f}%",
                  annotation_position="right",
                  annotation_font=dict(color=col, size=11))
    fig.add_scatter(
        x=[impl_vol], y=[mkt_price], mode="markers",
        marker=dict(color=col, size=10, symbol="circle"),
        name=f"IV ≈ {impl_vol:.0f}%", showlegend=True,
        hovertemplate=f"<b>{label}</b><br>Vol implicite: {impl_vol:.0f}%<extra></extra>",
    )

fig.update_layout(
    title="Inversion : prix de marché → vol implicite  (call ATM, T=90j)",
    xaxis_title="Volatilité implicite (%)", yaxis_title="Prix du call", height=380,
)
fig.show()

**Interprétation.** Deux options sur le même sous-jacent avec des strikes différents ont des
prix absolus incomparables — mais leurs vol implicites le sont. C'est pourquoi les traders
cotent et comparent tout en vol implicite : c'est la seule métrique qui met toutes les options
sur un pied d'égalité.

## 1.5 — Pourquoi la vol implicite n'est pas plate : trois raisons de marché

Dans un monde Black-Scholes pur, la vol serait identique pour tous les strikes et toutes les
maturités. En réalité elle ne l'est jamais — pour trois raisons structurelles.

In [ ]:
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
    subplot_titles=(
        "1. Queues épaisses vs normale",
        "2. Demande structurelle de puts",
        "3. Clusters de volatilité",
    ))

# ── Raison 1 : fat tails ──────────────────────────────────────────────────
rng2 = np.random.default_rng(0)
empirical_rets = np.concatenate([
    rng2.normal(0, 0.012, 800),
    rng2.standard_t(df=3, size=200) * 0.015,   # fat tails via Student-t
])
x_norm = np.linspace(-0.06, 0.06, 200)
fig.add_trace(go.Histogram(
    x=empirical_rets, nbinsx=60, histnorm="probability density",
    marker_color=C["blue"], opacity=0.7, name="Empirique",
    hovertemplate="Rendement: %{x:.2%}<br>Densité: %{y:.1f}<extra></extra>",
), row=1, col=1)
from scipy.stats import norm as sp_norm
fig.add_trace(go.Scatter(
    x=x_norm, y=sp_norm.pdf(x_norm, 0, empirical_rets.std()),
    mode="lines", line=dict(color=C["red"], width=2, dash="dash"),
    name="Loi normale",
    hovertemplate="Rendement: %{x:.2%}<br>Densité: %{y:.1f}<extra></extra>",
), row=1, col=1)

# ── Raison 2 : demande structurelle de puts ────────────────────────────────
strikes_dem = np.linspace(70, 110, 50)
demand_curve = 120 * np.exp(-0.08 * (strikes_dem - 70))
fig.add_trace(go.Scatter(
    x=strikes_dem, y=demand_curve, mode="lines+markers",
    line=dict(color=C["violet"], width=2.5), marker=dict(size=4),
    name="Volume d'achat puts", showlegend=False,
    hovertemplate="Strike: %{x:.0f}<br>Volume: %{y:.0f}<extra></extra>",
), row=1, col=2)
fig.add_vline(x=100, line_dash="dot", line_color=C["slate4"], row=1, col=2)
fig.add_annotation(x=100, y=110, text="Spot", showarrow=False, row=1, col=2,
                   font=dict(size=10, color=C["slate6"]))

# ── Raison 3 : clustering de vol ──────────────────────────────────────────
rng3  = np.random.default_rng(7)
days3 = np.arange(500)
regimes = np.ones(500) * 0.12
regimes[120:160] = 0.45   # crise courte
regimes[300:380] = 0.55   # crise longue
daily_rets3 = rng3.normal(0, regimes / np.sqrt(252))
rv_30 = np.array([daily_rets3[max(0,i-20):i].std() * np.sqrt(252)
                  for i in range(1, 501)])
fig.add_trace(go.Scatter(
    x=days3, y=rv_30 * 100, mode="lines",
    line=dict(color=C["amber"], width=1.5),
    name="Vol réalisée 20j", showlegend=False,
    hovertemplate="Jour %{x}<br>RV: %{y:.0f}%<extra></extra>",
), row=1, col=3)
fig.add_hrect(y0=30, y1=70, fillcolor=C["red"], opacity=0.08, row=1, col=3)

fig.update_xaxes(title_text="Rendement journalier", row=1, col=1)
fig.update_xaxes(title_text="Strike", row=1, col=2)
fig.update_xaxes(title_text="Jours", row=1, col=3)
fig.update_yaxes(title_text="Densité", row=1, col=1)
fig.update_yaxes(title_text="Volume relatif", row=1, col=2)
fig.update_yaxes(title_text="Vol réalisée (%)", row=1, col=3)
fig.update_layout(title="Pourquoi la vol implicite n'est pas plate", height=380,
                  margin=dict(t=95))
fig.show()

**Interprétation.**
- **Queues épaisses** : les rendements réels ont des queues bien plus épaisses que la loi normale.
  Les krachs arrivent plus souvent que Black-Scholes ne le prédit — le marché le sait et survalorise
  les options OTM en conséquence.
- **Demande structurelle de puts** : les gérants d'actifs actions achètent des puts OTM de manière
  quasi-permanente pour se couvrir. Cette demande mécanique, indépendante du "prix juste", gonfle
  la vol implicite des ailes gauches.
- **Clustering de volatilité** : la vol n'est pas constante — elle passe par régimes. Le marché
  anticipe ces transitions et les intègre dans le prix des options, créant une term structure non-plate.

---
# Bloc 2 — Les Greeks : tableau de bord d'une position options

Les Greeks mesurent comment le prix d'une option réagit à chaque paramètre de marché. Ce sont
les chiffres qu'un trader lit en premier sur son book. Delta, Vega et Gamma sont directement
liés à la surface de volatilité — ils expliquent pourquoi la surface *intéresse* un trader avant
même d'apprendre à la lire. Theta et Rho complètent le tableau de bord.

## 2.1 — Delta : la directionnalité

Delta mesure la variation du prix de l'option pour une variation de 1 unité du sous-jacent.
Un call ATM a un Delta de ≈0.50 : si le spot monte de 1€, l'option gagne ~0.50€.

In [ ]:
strikes_rng = np.linspace(70, 130, 200)
S_ref, T_ref, r_ref, sig_ref = 100., 90/365, 0.03, 0.25
T_maturities = [7/365, 30/365, 90/365, 180/365]
T_mat_labels = ["7j", "30j", "90j", "180j"]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=("Delta vs Strike (T=90j, σ=25%)",
                    "Delta ATM vs Maturité (call & put)"))

# Delta vs strike
deltas_c = [bs(S_ref, K, T_ref, r_ref, sig_ref, "call")["delta"] for K in strikes_rng]
deltas_p = [bs(S_ref, K, T_ref, r_ref, sig_ref, "put")["delta"]  for K in strikes_rng]
fig.add_trace(go.Scatter(x=strikes_rng, y=deltas_c, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Call",
    hovertemplate="Strike: %{x:.0f}<br>Delta: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=strikes_rng, y=deltas_p, mode="lines",
    line=dict(color=C["teal"], width=2.5), name="Put",
    hovertemplate="Strike: %{x:.0f}<br>Delta: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_vline(x=S_ref, line_dash="dash", line_color=C["slate4"], row=1, col=1)
fig.add_hline(y=0,    line_color=C["slate4"], line_width=0.8,   row=1, col=1)
fig.add_hline(y=0.5,  line_dash="dot", line_color=C["slate4"],  row=1, col=1)
fig.add_hline(y=-0.5, line_dash="dot", line_color=C["slate4"],  row=1, col=1)

# Delta ATM vs maturité
T_range = np.linspace(1/365, 1.0, 200)
deltas_atm_c = [bs(S_ref, S_ref, T, r_ref, sig_ref, "call")["delta"] for T in T_range]
deltas_atm_p = [bs(S_ref, S_ref, T, r_ref, sig_ref, "put")["delta"]  for T in T_range]
fig.add_trace(go.Scatter(x=T_range*365, y=deltas_atm_c, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Call ATM", showlegend=False,
    hovertemplate="T: %{x:.0f}j<br>Delta: %{y:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=T_range*365, y=deltas_atm_p, mode="lines",
    line=dict(color=C["teal"], width=2.5), name="Put ATM", showlegend=False,
    hovertemplate="T: %{x:.0f}j<br>Delta: %{y:.3f}<extra></extra>"), row=1, col=2)
fig.add_hline(y=0.5,  line_dash="dot", line_color=C["slate4"], row=1, col=2)
fig.add_hline(y=-0.5, line_dash="dot", line_color=C["slate4"], row=1, col=2)

fig.update_xaxes(title_text="Strike", row=1, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=1, col=2)
fig.update_yaxes(title_text="Delta", row=1, col=1)
fig.update_layout(title="Delta — Sensibilité directionnelle au sous-jacent", height=400,
                  margin=dict(t=90))
fig.show()

**Interprétation.** Le profil en S du Delta dit tout : profondément ITM le call se comporte
comme le sous-jacent lui-même (Delta→1), profondément OTM il ne bouge presque plus (Delta→0).
L'ATM est le point d'inflexion à Delta ≈ 0,5 (un peu au-dessus avec le carry). Plus la maturité est courte, plus cette transition
est abrupte — un call ATM à 7 jours peut passer de Delta=0.1 à Delta=0.9 sur un simple mouvement
de quelques pourcents. **Lien surface** : le skew déforme ce profil — sur un marché avec skew
négatif fort, le Delta d'un put OTM est structurellement plus élevé qu'en monde flat.

## 2.2 — Vega : la sensibilité à la volatilité

Vega mesure le gain (ou la perte) pour une hausse de 1% de la volatilité implicite. C'est le
Greek central pour un trader de surface : **gérer une book d'options = gérer son exposition
Vega** par strike et par maturité.

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=("Vega vs Strike — plusieurs maturités",
                    "Vega ATM vs Maturité"))

for (T, label, color) in zip(T_maturities, T_mat_labels,
                               [C["red"], C["amber"], C["blue"], C["teal"]]):
    vegas = [bs(S_ref, K, T, r_ref, sig_ref)["vega"] for K in strikes_rng]
    fig.add_trace(go.Scatter(
        x=strikes_rng, y=vegas, mode="lines",
        line=dict(color=color, width=2), name=label,
        hovertemplate=f"<b>{label}</b><br>Strike: %{{x:.0f}}<br>Vega/1%: %{{y:.3f}}<extra></extra>",
    ), row=1, col=1)

fig.add_vline(x=S_ref, line_dash="dash", line_color=C["slate4"], row=1, col=1)

# Vega ATM vs maturité
vegas_atm = [bs(S_ref, S_ref, T, r_ref, sig_ref)["vega"] for T in T_range]
fig.add_trace(go.Scatter(
    x=T_range*365, y=vegas_atm, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Vega ATM", showlegend=False,
    hovertemplate="T: %{x:.0f}j<br>Vega/1%: %{y:.3f}<extra></extra>",
), row=1, col=2)

fig.update_xaxes(title_text="Strike", row=1, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=1, col=2)
fig.update_yaxes(title_text="Vega (par +1% de vol)", row=1, col=1)
fig.update_yaxes(title_text="Vega ATM", row=1, col=2)
fig.update_layout(title="Vega — Sensibilité à la volatilité implicite", height=400,
                  margin=dict(t=90))
fig.show()

**Interprétation.** Le Vega est maximal ATM et s'écrase aux ailes — les options ATM sont les
plus sensibles aux mouvements de vol implicite. Une position long Vega (acheteur net d'options)
gagne quand la surface "se soulève", short Vega quand elle s'aplatit. Le Vega croît avec la
racine de la maturité : une option à 180j est bien plus sensible à la vol qu'une option à 7j
— c'est là que les institutionnels expriment des vues sur la vol structurelle. **C'est le lien
direct avec la surface** : chaque cellule (strike, maturité) de la surface correspond à un bucket
de Vega.

## 2.3 — Gamma : la convexité

Gamma mesure la vitesse à laquelle le Delta change. Long Gamma = on bénéficie des grands
mouvements dans les deux sens (le Delta s'ajuste favorablement). C'est la convexité pure.

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=("Gamma vs Strike — par maturité",
                    "Gamma ATM vs Maturité"))

for T, label, color in zip(T_maturities, T_mat_labels,
                             [C["red"], C["amber"], C["blue"], C["teal"]]):
    gammas = [bs(S_ref, K, T, r_ref, sig_ref)["gamma"] for K in strikes_rng]
    fig.add_trace(go.Scatter(
        x=strikes_rng, y=gammas, mode="lines",
        line=dict(color=color, width=2), name=label,
        hovertemplate=f"<b>{label}</b><br>Strike: %{{x:.0f}}<br>Gamma: %{{y:.5f}}<extra></extra>",
    ), row=1, col=1)

fig.add_vline(x=S_ref, line_dash="dash", line_color=C["slate4"], row=1, col=1)

gammas_atm = [bs(S_ref, S_ref, T, r_ref, sig_ref)["gamma"] for T in T_range]
fig.add_trace(go.Scatter(
    x=T_range*365, y=gammas_atm, mode="lines",
    line=dict(color=C["blue"], width=2.5), showlegend=False,
    hovertemplate="T: %{x:.0f}j<br>Gamma: %{y:.5f}<extra></extra>",
), row=1, col=2)

fig.update_xaxes(title_text="Strike", row=1, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=1, col=2)
fig.update_yaxes(title_text="Gamma", row=1, col=1)
fig.update_yaxes(title_text="Gamma ATM", row=1, col=2)
fig.update_layout(title="Gamma — Convexité de la position", height=400,
                  margin=dict(t=90))
fig.show()

**Interprétation.** Le Gamma explose à l'approche de l'expiry ATM : une option qui expire demain
à la monnaie peut voir son Delta passer de 0 à 1 sur un mouvement minime — c'est un risque de
discontinuité. Les options courte maturité ATM offrent le plus de convexité : c'est pourquoi
elles sont chères en termes de Gamma. **Lien surface** : la courbure en U du smile (convexity)
est la traduction directe du Gamma de marché — un smile très courbé signifie que le marché
valorise beaucoup de convexité. (Sur le panneau « Gamma ATM vs maturité », « proche de l'expiry » = petites maturités, à gauche.)

## 2.4 — Theta : le coût du temps

Theta mesure l'érosion journalière de la valeur temps. **Note** : Theta n'est pas directement lié
à la surface de volatilité, mais il est la contrepartie mécanique du Gamma — long Gamma = short
Theta. On paie le temps pour avoir la convexité (valable proche de la monnaie).

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=("Theta vs Strike (T=90j)",
                    "Theta ATM vs Maturité"))

thetas_c = [bs(S_ref, K, T_ref, r_ref, sig_ref, "call")["theta"] for K in strikes_rng]
thetas_p = [bs(S_ref, K, T_ref, r_ref, sig_ref, "put")["theta"]  for K in strikes_rng]
fig.add_trace(go.Scatter(x=strikes_rng, y=thetas_c, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Call",
    hovertemplate="Strike: %{x:.0f}<br>Theta/j: %{y:.4f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=strikes_rng, y=thetas_p, mode="lines",
    line=dict(color=C["teal"], width=2.5), name="Put",
    hovertemplate="Strike: %{x:.0f}<br>Theta/j: %{y:.4f}<extra></extra>"), row=1, col=1)
fig.add_vline(x=S_ref, line_dash="dash", line_color=C["slate4"], row=1, col=1)
fig.add_hline(y=0, line_color=C["slate4"], line_width=0.8, row=1, col=1)

T_range2 = np.linspace(2/365, 1.0, 200)
thetas_atm = [bs(S_ref, S_ref, T, r_ref, sig_ref, "call")["theta"] for T in T_range2]
fig.add_trace(go.Scatter(x=T_range2*365, y=thetas_atm, mode="lines",
    line=dict(color=C["blue"], width=2.5), showlegend=False,
    hovertemplate="T: %{x:.0f}j<br>Theta/j: %{y:.4f}<extra></extra>"), row=1, col=2)
fig.add_hline(y=0, line_color=C["slate4"], line_width=0.8, row=1, col=2)

fig.update_xaxes(title_text="Strike", row=1, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=1, col=2)
fig.update_yaxes(title_text="Theta (par jour)", row=1, col=1)
fig.update_layout(title="Theta — Érosion journalière de la valeur temps", height=400,
                  margin=dict(t=90))
fig.show()

**Interprétation.** Le Theta est maximal (en valeur absolue) ATM et s'accélère fortement à
l'approche de l'expiry — les derniers jours de vie d'une option voient une érosion de valeur
temps rapide. C'est le prix à payer pour la convexité (Gamma) : une stratégie long Gamma/short
Theta est rentable si les mouvements réalisés compensent le saignement de Theta. C'est le
compromis fondamental de tout book d'options.

## 2.5 — Rho : la sensibilité aux taux

Rho mesure l'impact d'une variation de 1bp du taux sans risque. **Note** : c'est le Greek le
moins utilisé au quotidien sauf dans des environnements de taux très mobiles (2022–2023). Sur
des options courte maturité, son impact est marginal.

In [ ]:
rates = np.linspace(0.0, 0.08, 200)
prices_r_c = [bs(S_ref, S_ref, T_ref, r, sig_ref, "call")["price"] for r in rates]
prices_r_p = [bs(S_ref, S_ref, T_ref, r, sig_ref, "put")["price"]  for r in rates]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=("Prix ATM vs Taux sans risque",
                    "Rho vs Strike (T=90j)"))

fig.add_trace(go.Scatter(x=rates*100, y=prices_r_c, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Call ATM",
    hovertemplate="Taux: %{x:.1f}%<br>Prix: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=rates*100, y=prices_r_p, mode="lines",
    line=dict(color=C["teal"], width=2.5), name="Put ATM",
    hovertemplate="Taux: %{x:.1f}%<br>Prix: %{y:.3f}<extra></extra>"), row=1, col=1)

rhos_c = [bs(S_ref, K, T_ref, r_ref, sig_ref, "call")["rho"] for K in strikes_rng]
rhos_p = [bs(S_ref, K, T_ref, r_ref, sig_ref, "put")["rho"]  for K in strikes_rng]
fig.add_trace(go.Scatter(x=strikes_rng, y=rhos_c, mode="lines",
    line=dict(color=C["blue"], width=2.5), name="Call", showlegend=False,
    hovertemplate="Strike: %{x:.0f}<br>Rho/bp: %{y:.6f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=strikes_rng, y=rhos_p, mode="lines",
    line=dict(color=C["teal"], width=2.5), name="Put", showlegend=False,
    hovertemplate="Strike: %{x:.0f}<br>Rho/bp: %{y:.6f}<extra></extra>"), row=1, col=2)
fig.add_vline(x=S_ref, line_dash="dash", line_color=C["slate4"], row=1, col=2)
fig.add_hline(y=0, line_color=C["slate4"], line_width=0.8, row=1, col=2)

fig.update_xaxes(title_text="Taux sans risque (%)", row=1, col=1)
fig.update_xaxes(title_text="Strike", row=1, col=2)
fig.update_yaxes(title_text="Prix", row=1, col=1)
fig.update_yaxes(title_text="Rho (par +1bp)", row=1, col=2)
fig.update_layout(title="Rho — Sensibilité aux taux (marginal sur courte maturité)", height=380,
                  margin=dict(t=90))
fig.show()

**Interprétation.** Les calls bénéficient d'une hausse des taux (le coût de portage du sous-jacent
augmente, le call vaut davantage), les puts en souffrent. L'effet est monotone mais faible sur
courte maturité — quelques centimes pour 100bp. Il devient matériel sur des options longue durée
(LEAPS à 1-2 ans) ou dans des environnements où les banques centrales bougent agressivement.

## 2.6 — Tableau de bord Greeks : récapitulatif

Un long call sur un actif référence. Ce tableau de bord est ce qu'on lit en premier sur un
risk report — savoir en 30 secondes si un book est long ou short chaque Greek.

In [ ]:
T_db = np.linspace(2/365, 0.6, 150)
greek_data = {
    "Prix"  : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["price"] for T in T_db], C["blue"],  "Prix"),
    "Delta" : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["delta"] for T in T_db], C["teal"],  "Delta"),
    "Gamma" : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["gamma"] for T in T_db], C["violet"],"Gamma"),
    "Vega"  : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["vega"]  for T in T_db], C["amber"], "Vega/1%"),
    "Theta" : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["theta"] for T in T_db], C["red"],   "Theta/j"),
    "Rho"   : ([bs(S_ref, S_ref, T, r_ref, sig_ref)["rho"]   for T in T_db], C["indigo"],"Rho/bp"),
}
signs = {"Prix": "⊕", "Delta": "⊕ [0,1]", "Gamma": "⊕", "Vega": "⊕", "Theta": "⊖", "Rho": "⊕"}

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.08, vertical_spacing=0.15,
    subplot_titles=[f"{g}  ({s})" for g, s in signs.items()])
pos = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for (name, (vals, color, ylabel)), (r, c) in zip(greek_data.items(), pos):
    fig.add_trace(go.Scatter(
        x=T_db*365, y=vals, mode="lines",
        line=dict(color=color, width=2.5), name=name, showlegend=False,
        hovertemplate=f"<b>{name}</b><br>T: %{{x:.0f}}j<br>{ylabel}: %{{y:.4f}}<extra></extra>",
    ), row=r, col=c)
    fig.add_hline(y=0, line_color=C["slate4"], line_width=0.6, row=r, col=c)
    fig.update_yaxes(title_text=ylabel, row=r, col=c)

fig.update_xaxes(title_text="Maturité (jours)", row=2, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=2, col=2)
fig.update_xaxes(title_text="Maturité (jours)", row=2, col=3)
fig.update_layout(
    title="Tableau de bord — Long call ATM (S=K=100, σ=25%, r=3%)  ⊕=positif  ⊖=négatif",
    height=560,
    margin=dict(t=100),
)
fig.show()

**Interprétation.** Un long call ATM est : long Delta (profite de la hausse), long Gamma
(profite des grands mouvements), long Vega (profite de la hausse de vol), short Theta (perd
de la valeur chaque jour), et légèrement long Rho (profite des hausses de taux). Ce tableau de
bord est symétrique : un short call inverse tous les signes. La gestion d'un book d'options,
c'est l'art d'équilibrer ces expositions selon la vue de marché et le budget de risque. (Tous les Greeks sont tracés en fonction de la maturité, court terme à gauche : Gamma et Theta y sont les plus extrêmes près de l'expiry.)

---
# Bloc 3 — La tranche : lire un smile à maturité fixée (2D)

Avant de construire la surface 3D, il faut maîtriser la lecture d'une coupe horizontale —
la vol implicite pour tous les strikes à une maturité donnée.

## 3.1 — La vol plate : le monde Black-Scholes

Si Black-Scholes était vrai, la vol implicite serait identique pour tous les strikes.

In [ ]:
smiles = {
    "Plate (Black-Scholes)": smile_iv(k_grid, atm=0.25, skew=0.0,  conv=0.0),
    "Smile symétrique (crypto)": smile_iv(k_grid, atm=0.30, skew=0.0,  conv=2.0),
    "Skew négatif (actions)": smile_iv(k_grid, atm=0.22, skew=-0.40, conv=0.5),
    "Skew positif (matières 1ères)": smile_iv(k_grid, atm=0.28, skew=0.50,  conv=0.8),
}
smile_colors = [C["slate4"], C["teal"], C["blue"], C["amber"]]

fig = go.Figure()
for (label, iv), color in zip(smiles.items(), smile_colors):
    fig.add_trace(go.Scatter(
        x=k_grid, y=iv*100, mode="lines",
        line=dict(color=color, width=2.5 if label != "Plate (Black-Scholes)" else 1.5,
                  dash="dash" if "Plate" in label else "solid"),
        name=label,
        hovertemplate=f"<b>{label}</b><br>k: %{{x:.3f}}<br>IV: %{{y:.1f}}%<extra></extra>",
    ))
fig.add_vline(x=0, line_dash="dot", line_color=C["slate4"],
              annotation_text="ATM", annotation_position="top")
fig.update_layout(
    title="Les quatre formes caractéristiques du smile — IV vs log-moneyness (k = ln K/F)",
    xaxis_title="Log-moneyness  k = ln(K/F)",
    yaxis_title="Volatilité implicite (%)",
    height=420,
    margin=dict(t=80),
)
fig.show()

**Interprétation.** Ce plot superpose les quatre formes fondamentales.

- **Plate** : le cas théorique Black-Scholes. N'existe pas en pratique — c'est la référence visuelle.
- **Smile symétrique** : le marché price des sauts violents dans les deux sens. Typique du BTC/ETH
  ou d'un actif avant un événement binaire (résultat trimestriel, vote clé). La peur est équilibrée.
- **Skew négatif** : les puts OTM sont structurellement chers, les calls bon marché. Typique des
  indices actions. La demande de protection crash des institutionnels crée cette asymétrie permanente.
- **Skew positif** : les calls OTM sont chers. Typique du pétrole, du gaz — le marché craint un
  rally violent (choc d'offre, embargo) bien plus qu'une baisse.

## 3.2 — Les trois paramètres du smile

Le smile se lit en trois paramètres : le **niveau ATM** (la peur absolue), la **pente** (la
direction de la peur), et la **courbure** (l'intensité du jump risk).

In [ ]:
iv_annotated = smile_iv(k_grid, atm=0.22, skew=-0.40, conv=0.5)
k_atm_idx = np.argmin(np.abs(k_grid))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=k_grid, y=iv_annotated*100, mode="lines",
    line=dict(color=C["blue"], width=3), name="Skew négatif (actions)",
    hovertemplate="k: %{x:.3f}<br>IV: %{y:.1f}%<extra></extra>",
))
fig.add_vline(x=0, line_dash="dot", line_color=C["slate4"])

# Annotation 1 : niveau ATM
atm_val = iv_annotated[k_atm_idx] * 100
fig.add_annotation(x=0, y=atm_val, text=f"Niveau ATM = {atm_val:.0f}%<br>← Peur absolue (≈VIX)",
    showarrow=True, arrowhead=2, ax=80, ay=-30,
    font=dict(size=11, color=C["blue"]), arrowcolor=C["blue"])

# Annotation 2 : pente (skew)
fig.add_annotation(
    x=-0.3, y=smile_iv(-0.3, 0.22, -0.40, 0.5)*100,
    text="Pente gauche ↑<br>← Direction de la peur<br>(crash > rally)",
    showarrow=True, arrowhead=2, ax=-60, ay=-40,
    font=dict(size=11, color=C["teal"]), arrowcolor=C["teal"])

# Annotation 3 : courbure
fig.add_annotation(
    x=0.35, y=smile_iv(0.35, 0.22, -0.40, 0.5)*100,
    text="Courbure ↗<br>← Intensité du jump risk<br>(queues épaisses)",
    showarrow=True, arrowhead=2, ax=60, ay=-30,
    font=dict(size=11, color=C["violet"]), arrowcolor=C["violet"])

fig.update_layout(
    title="Les trois paramètres du smile — Niveau ATM / Pente (skew) / Courbure (convexité)",
    xaxis_title="Log-moneyness  k = ln(K/F)",
    yaxis_title="Volatilité implicite (%)",
    height=420,
    margin=dict(t=80),
)
fig.show()

**Interprétation.** Ces trois paramètres résument l'intégralité du consensus de marché sur le
risque à cette maturité. Un senior trader lit ce graphe en quelques secondes : ici le marché
craint modérément un crash (niveau ATM 22%, raisonnable), la peur est nettement asymétrique
vers le bas (pente fortement négative), et le jump risk est présent mais modéré (courbure faible
à droite). **Lien Greeks** : la pente correspond au skew de Delta (les puts OTM ont un Delta
plus élevé que prédit par BS), et la courbure correspond au Gamma de marché (plus c'est courbé,
plus le marché price de la convexité).

---
# Bloc 4 — La surface complète : empiler les tranches en 3D

La surface de volatilité n'est pas un objet mystérieux : c'est l'empilement des smiles vus en
Bloc 3 sur l'axe maturité.

## 4.1 — De la tranche à la surface

On part de 4 smiles à maturités différentes, on les superpose, puis on bascule en 3D.

In [ ]:
# 4 tranches de la surface equity
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
    subplot_titles=("4 tranches superposées (2D)", "La même surface en 3D"),
    specs=[[{"type": "xy"}, {"type": "scene"}]])

slice_colors = [C["red"], C["amber"], C["blue"], C["teal"]]
for iv_slice, label, color in zip(surf_equity, T_labels, slice_colors):
    fig.add_trace(go.Scatter(
        x=k_grid, y=iv_slice*100, mode="lines",
        line=dict(color=color, width=2), name=label,
        hovertemplate=f"<b>{label}</b><br>k: %{{x:.3f}}<br>IV: %{{y:.1f}}%<extra></extra>",
    ), row=1, col=1)

fig.add_vline(x=0, line_dash="dot", line_color=C["slate4"], row=1, col=1)

fig.add_trace(go.Surface(
    x=KK, y=TT, z=surf_equity*100,
    colorscale=SURFACE_CS, opacity=0.88,
    contours=dict(z=dict(show=True, usecolormap=True, project_z=True)),
    colorbar=dict(title="IV%", thickness=14, len=0.6, x=1.02,
                  tickformat=".0f"),
    hovertemplate="k: %{x:.3f}<br>T: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
    showscale=True, name="Surface equity",
), row=1, col=2)

fig.update_xaxes(title_text="Log-moneyness k", row=1, col=1)
fig.update_yaxes(title_text="IV (%)", row=1, col=1)
fig.update_layout(
    title="De la tranche à la surface — Surface equity réaliste",
    height=520,
    margin=dict(t=90),
    scene=dict(
        xaxis=dict(title="Log-moneyness", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        yaxis=dict(title="Maturité (j)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        zaxis=dict(title="IV (%)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        camera=CAMERA, aspectmode="manual", aspectratio=ASPECT,
    ),
)
fig.show()

**Interprétation.** Le plot 2D montre clairement l'aplatissement progressif du smile avec la
maturité : la tranche rouge (30j) a un skew très prononcé, la verte (180j) est bien plus plate.
En 3D, cet aplatissement se lit comme une surface qui "s'écrase" vers le fond. Les contours
projetés au sol permettent de lire le skew sans avoir à tourner la vue.

## 4.2 — La term structure : trois régimes

La term structure décrit comment la vol ATM évolue avec la maturité. Elle révèle ce que le
marché anticipe pour les prochaines semaines vs les prochains mois.

In [ ]:
T_fine_arr = np.linspace(1/365, 0.6, 60)

surf_back = build_surface(T_fine_arr, k_grid, ts_backwardation,
                           base_skew=-0.30, base_conv=0.30, skew_fade=2., conv_fade=1.5)
surf_flat_ts = build_surface(T_fine_arr, k_grid, ts_flat,
                              base_skew=-0.20, base_conv=0.20)
surf_cont = build_surface(T_fine_arr, k_grid, ts_contango,
                           base_skew=-0.20, base_conv=0.20, skew_fade=2., conv_fade=1.5)
KK2, TT2 = np.meshgrid(k_grid, T_fine_arr * 365)

ts_titles = [
    "Backwardation (vol courte > longue)",
    "Flat (pas de vue temporelle)",
    "Contango (vol courte < longue)",
]
ts_surfs = [surf_back, surf_flat_ts, surf_cont]

fig = make_subplots(rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=ts_titles, horizontal_spacing=0.06)

for col, surf in enumerate(ts_surfs, 1):
    fig.add_trace(go.Surface(
        x=KK2, y=TT2, z=surf*100,
        colorscale=SURFACE_CS, opacity=0.88, showscale=(col==3),
        contours=dict(z=dict(show=True, usecolormap=True, project_z=True)),
        colorbar=dict(title="IV%", thickness=12, len=0.5, x=1.05, tickformat=".0f"),
        hovertemplate="k: %{x:.3f}<br>T: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
        name=ts_titles[col-1][:20],
    ), row=1, col=col)
    scene_key = f"scene{'' if col == 1 else col}"
    cam = dict(eye=dict(x=1.5, y=-1.5, z=0.8), up=dict(x=0, y=0, z=1))
    fig.update_layout(**{scene_key: dict(
        xaxis=dict(title="k", backgroundcolor=C["slate1"], showbackground=True),
        yaxis=dict(title="T (j)", backgroundcolor=C["slate1"], showbackground=True),
        zaxis=dict(title="IV%", backgroundcolor=C["slate1"], showbackground=True,
                   range=[0, 80]),
        camera=cam, aspectmode="manual",
        aspectratio=dict(x=1.2, y=1.0, z=0.6),
    )})

fig.update_layout(
    title="Term structure : trois régimes — même skew, term structure différente",
    height=520,
    margin=dict(t=95, r=80),
)
fig.show()

**Interprétation.**
- **Backwardation** : la surface plonge vers l'arrière — la vol courte terme est bien supérieure
  à la vol longue terme. Signal clair d'un événement de risque imminent : annonce Fed, résultats,
  vote politique. Le marché anticipe un choc à court terme puis un retour au calme.
- **Flat** : aucune vue temporelle. Soit le marché est indifférent, soit il manque de liquidité
  sur certaines maturités pour établir une structure.
- **Contango** : la surface monte vers l'arrière. Le marché est serein maintenant mais incertain
  structurellement à long terme. C'est le régime "normal" des actions en dehors des crises.

## 4.3 — Surface réaliste actions

Combine : skew négatif fort court terme + contango ATM + aplatissement progressif avec la
maturité. C'est la forme la plus commune sur les indices actions (SPY, Euro Stoxx 50).

In [ ]:
fig = go.Figure(go.Surface(
    x=KK, y=TT, z=surf_equity*100,
    colorscale=SURFACE_CS, opacity=0.88,
    lighting=dict(ambient=0.7, diffuse=0.6, roughness=0.5, specular=0.1),
    contours=dict(z=dict(show=True, usecolormap=True,
                         highlightcolor=C["white"], project_z=True)),
    colorbar=dict(title=dict(text="IV (%)", side="right"),
                  tickformat=".0f", thickness=16, len=0.7, x=1.02),
    hovertemplate="log-m: %{x:.3f}<br>Maturité: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
))
fig.update_layout(
    title="Surface réaliste actions — Skew négatif + contango + aplatissement LT",
    height=620,
    scene=dict(
        xaxis=dict(title="Log-Moneyness", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        yaxis=dict(title="Maturité (jours)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        zaxis=dict(title="IV (%)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        camera=CAMERA, aspectmode="manual", aspectratio=ASPECT,
    ),
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()

**Interprétation.** C'est la surface que l'on observe sur SPY ou l'Euro Stoxx 50 en régime
normal. Trois caractéristiques simultanées : (1) le flanc gauche est nettement plus haut que
le flanc droit — skew de crash permanent dû à la demande institutionnelle de puts ; (2) la
vol ATM monte avec la maturité (contango) — le marché est relativement serein court terme mais
incertain long terme ; (3) la surface s'aplatit vers le fond — le skew se comprime avec l'horizon
car la distribution des rendements redevient plus symétrique à mesure que l'horizon s'allonge.

## 4.4 — Surface réaliste crypto

Combine : smile symétrique fort + backwardation courte terme + aplatissement des ailes long terme.
Typique du BTC/ETH en régime incertain.

In [ ]:
fig = go.Figure(go.Surface(
    x=KK, y=TT, z=surf_crypto*100,
    colorscale=SURFACE_CS, opacity=0.88,
    lighting=dict(ambient=0.7, diffuse=0.6, roughness=0.5, specular=0.1),
    contours=dict(z=dict(show=True, usecolormap=True,
                         highlightcolor=C["white"], project_z=True)),
    colorbar=dict(title=dict(text="IV (%)", side="right"),
                  tickformat=".0f", thickness=16, len=0.7, x=1.02),
    hovertemplate="log-m: %{x:.3f}<br>Maturité: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
))
fig.update_layout(
    title="Surface réaliste crypto (BTC) — Smile symétrique + backwardation + aplatissement LT",
    height=620,
    scene=dict(
        xaxis=dict(title="Log-Moneyness", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        yaxis=dict(title="Maturité (jours)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        zaxis=dict(title="IV (%)", gridcolor=C["slate4"],
                   backgroundcolor=C["slate1"], showbackground=True),
        camera=CAMERA, aspectmode="manual", aspectratio=ASPECT,
    ),
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()

**Interprétation.** La différence avec la surface actions est immédiatement lisible : (1) la
surface est quasi-symétrique gauche/droite — pas de biais directionnel, le marché craint autant
un crash qu'un rally violent ; (2) la vol est maximale court terme (backwardation) — le marché
perçoit une agitation immédiate qui se dissipera ; (3) les ailes s'aplatissent fortement avec
l'horizon. Cette forme est typique d'un actif spéculatif en régime de forte incertitude directionnelle.

## 4.5 — Violations calendaires

La variance totale `σ²·T` doit être **strictement croissante** avec la maturité. Si elle décroît,
un arbitrage calendaire est théoriquement possible. En pratique c'est un signal d'illiquidité
ou d'incohérence de cotation — le premier flag à vérifier sur une surface.

In [ ]:
# Identifier les violations sur surf_arb
violations = []
for ki, k in enumerate(k_grid):
    for ti in range(len(T_arr) - 1):
        tv_lo = surf_arb[ti,   ki]**2 * T_arr[ti]
        tv_hi = surf_arb[ti+1, ki]**2 * T_arr[ti+1]
        if tv_hi < tv_lo - 1e-6:
            violations.append((k, T_arr[ti+1]*365,
                                surf_arb[ti+1, ki]*100))

vx = [v[0] for v in violations]
vy = [v[1] for v in violations]
vz = [v[2] for v in violations]

fig = go.Figure()
fig.add_trace(go.Surface(
    x=KK, y=TT, z=surf_arb*100,
    colorscale=SURFACE_CS, opacity=0.80,
    contours=dict(z=dict(show=True, usecolormap=True, project_z=True)),
    colorbar=dict(title=dict(text="IV (%)", side="right"),
                  tickformat=".0f", thickness=14, len=0.6, x=1.02),
    hovertemplate="k: %{x:.3f}<br>T: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
    name="Surface",
))
if violations:
    fig.add_trace(go.Scatter3d(
        x=vx, y=vy, z=vz, mode="markers",
        marker=dict(color=C["red"], size=5, symbol="x",
                    line=dict(color=C["white"], width=1)),
        name=f"Violations calendaires ({len(violations)})",
        hovertemplate="<b>VIOLATION</b><br>k: %{x:.3f}<br>T: %{y:.0f}j<br>IV: %{z:.1f}%<extra></extra>",
    ))

fig.update_layout(
    title=f"Surface avec violations calendaires — {len(violations)} points en rouge (σ²·T non-croissant)",
    height=620,
    scene=dict(
        xaxis=dict(title="Log-Moneyness", backgroundcolor=C["slate1"], showbackground=True),
        yaxis=dict(title="Maturité (jours)", backgroundcolor=C["slate1"], showbackground=True),
        zaxis=dict(title="IV (%)", backgroundcolor=C["slate1"], showbackground=True),
        camera=CAMERA, aspectmode="manual", aspectratio=ASPECT,
    ),
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()
print(f"Violations détectées : {len(violations)} points  (slice 60j sous la slice 30j)")

**Interprétation.** Les croix rouges marquent les points où la variance totale `σ²·T` décroît
de la slice 30j à la slice 60j — ce qui signifie qu'une option à 60j est moins chère en variance
qu'une option à 30j au même log-moneyness (forward-moneyness comparable, pas le même strike dollar). Théoriquement, un trader pourrait acheter le spread
calendaire pour encaisser la différence sans risque. En pratique, ces violations apparaissent
sur des strikes très OTM peu liquides, ou quand deux teneurs de marché différents cotent les
deux maturités. C'est le premier contrôle à faire sur toute surface avant de l'utiliser pour pricer.

## 4.6 — Coupes pratiques : lire la surface en 30 secondes

La surface 3D est la vision globale. Les coupes 2D sont l'outil d'exécution : on lit le skew
d'une maturité précise (coupe horizontale) ou la dynamique temporelle d'un strike précis
(coupe verticale).

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=(
        "Coupes horizontales — Smile par maturité",
        "Coupes verticales — Term structure par strike",
    ))

# Coupes horizontales
for iv_slice, label, color in zip(surf_equity, T_labels, slice_colors):
    fig.add_trace(go.Scatter(
        x=k_grid, y=iv_slice*100, mode="lines",
        line=dict(color=color, width=2.5), name=label,
        hovertemplate=f"<b>{label}</b><br>k: %{{x:.3f}}<br>IV: %{{y:.1f}}%<extra></extra>",
    ), row=1, col=1)
fig.add_vline(x=0, line_dash="dot", line_color=C["slate4"], row=1, col=1)

# Coupes verticales
k_cuts = {"OTM put  k=-0.30": -0.30, "ATM  k=0.00": 0.00, "OTM call  k=+0.30": 0.30}
cut_colors = [C["blue"], C["teal"], C["amber"]]
T_days_arr = T_arr * 365

for (label, k_val), color in zip(k_cuts.items(), cut_colors):
    ki = np.argmin(np.abs(k_grid - k_val))
    ts_ivs = surf_equity[:, ki] * 100
    fig.add_trace(go.Scatter(
        x=T_days_arr, y=ts_ivs, mode="lines+markers",
        line=dict(color=color, width=2.5), marker=dict(size=8),
        name=label, showlegend=True,
        hovertemplate=f"<b>{label}</b><br>T: %{{x:.0f}}j<br>IV: %{{y:.1f}}%<extra></extra>",
    ), row=1, col=2)

fig.update_xaxes(title_text="Log-moneyness k", row=1, col=1)
fig.update_xaxes(title_text="Maturité (jours)", row=1, col=2)
fig.update_yaxes(title_text="IV (%)", row=1, col=1)
fig.update_yaxes(title_text="IV (%)", row=1, col=2)
fig.update_layout(
    title="Lecture pratique de la surface — Coupes horizontales et verticales",
    height=430,
    margin=dict(t=95),
)
fig.show()

**Interprétation.**

**Coupes horizontales** : l'aplatissement progressif du smile avec la maturité est immédiatement
lisible — la tranche 30j (rouge) a un skew très prononcé, la 180j (verte) est quasiment plate.
Cela dit que la peur de crash est principalement un phénomène court terme : les institutionnels
se couvrent sur les prochains 30–60 jours, pas sur 6 mois.

**Coupes verticales** : chaque strike a sa propre term structure. Le put OTM (bleu) a une vol
élevée court terme qui décline — c'est le skew qui s'aplatit. L'ATM (teal) monte légèrement
avec la maturité (contango). Le call OTM (ambre) part bas et reste bas — les calls lointains
sont structurellement bon marché. **Lien Vega** : gérer la surface, c'est gérer ces trois courbes
simultanément — chacune représente un bucket d'exposition Vega dans le book.

---
## Récapitulatif — Ce qu'il faut retenir

| Concept | En une phrase |
|---|---|
| **Vol implicite** | Le seul paramètre d'information dans le prix d'une option — le consensus du marché sur l'agitation future |
| **Delta** | Exposition directionnelle — combien bouge l'option quand le spot bouge de 1 |
| **Vega** | Exposition à la vol — combien l'option gagne pour +1% de vol implicite |
| **Gamma** | Convexité — long Gamma profite des grands mouvements, mais saigne du Theta |
| **Theta** | Coût du temps — la contrepartie du Gamma, s'accélère à l'approche de l'expiry |
| **Rho** | Sensibilité aux taux — marginal sur courte maturité, matériel sur LEAPS |
| **Smile plat** | Monde Black-Scholes — n'existe pas en pratique |
| **Smile symétrique** | Marché bimodal, incertitude directionnelle — crypto, événement binaire |
| **Skew négatif** | Peur de crash structurelle — indices actions, demande institutionnelle de puts |
| **Skew positif** | Peur de rally — matières premières, risque de squeeze |
| **Backwardation** | Événement de risque imminent — la vol court terme > long terme |
| **Contango** | Régime normal — sérénité CT, incertitude LT |
| **Violation calendaire** | σ²·T décroissant — flag d'illiquidité ou d'incohérence de cotation |

**Pour aller plus loin avec le moteur :**
- `notebooks/demo_pipeline_deribit_v2.ipynb` — pipeline complet sur vraies options BTC Deribit
- `packages/infra/docs/blueprint/` — référence mathématique complète (formules, contrats de données)